# SwiftCart Online Retail Analytics using Python
## Sprint 3 – Advanced Statistics & Feature Engineering

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
sns.set_theme(style="whitegrid")

In [2]:
df = pd.read_csv("SwiftCart_Cleaned_Dataset.csv", parse_dates=["Date"])
df.head()

,Order ID,Cust ID,Gender,Age,Age Group,Date,Month,Status,Channel,SKU,Category,Size,Qty,currency,Amount,ship-city,ship-state,ship-postal-code,ship-country,B2B
0,171-1029312-3038738,"1,029,312.00",Women,NaN,Adult,2022-12-04,Dec,Delivered,Myntra,JNE1233-BLUE-KR-031-XXL,kurta,XXL,1,INR,376,MOHALI,PUNJAB,140301,IN,False
1,405-2183842-2225946,"2,183,842.00",Women,29.00,Teenager,2022-12-04,Dec,Delivered,Ajio,SET414-KR-NP-L,Set,L,1,INR,1449,GURUGRAM,HARYANA,122002,IN,False
2,171-1641533-8921966,"1,641,533.00",Women,67.00,Senior,2022-12-04,Dec,Delivered,Myntra,SET261-KR-PP-S,Set,S,1,INR,453,KOLKATA,WEST BENGAL,700029,IN,False
3,404-7490807-6300351,"7,490,807.00",Women,20.00,Teenager,2022-12-04,Dec,Delivered,Amazon,SET110-KR-PP-M,Set,M,1,INR,729,THANJAVUR,TAMIL NADU,613007,IN,False
4,403-9293516-4577154,"9,293,516.00",Women,62.00,Senior,2022-12-04,Dec,Delivered,Myntra,JNE2294-KR-A-XXL,kurta,XXL,1,INR,544,GURUGRAM,HARYANA,122001,IN,False


## 1. Measures of Central Tendency, Dispersion and Percentiles

### Question: What do the central tendency, dispersion, and percentiles show about Amount?

In [3]:
stats_summary = df["Amount"].agg(["mean","median","std","var","min","max"])
percentiles = df["Amount"].quantile([.01,.25,.50,.75,.90,.95,.99])
stats_summary, percentiles

(mean        682.07
 median      646.00
 std         268.58
 var      72,136.38
 min         229.00
 max       3,036.00
 Name: Amount, dtype: float64,
 0.01     292.00
 0.25     486.00
 0.50     646.00
 0.75     794.00
 0.90   1,099.00
 0.95   1,186.00
 0.99   1,442.00
 Name: Amount, dtype: float64)

**Observation:** The mean is higher than the median, which is consistent with a right-skewed sales distribution. The spread between the lower and upper percentiles shows that transaction amounts vary meaningfully.

## 2. Distribution Analysis

### Question: What are the skewness and kurtosis of sales amount?

In [4]:
skewness = stats.skew(df["Amount"], bias=False)
kurtosis = stats.kurtosis(df["Amount"], bias=False)
pd.Series({"Skewness": skewness, "Excess Kurtosis": kurtosis})

Skewness          1.05
Excess Kurtosis   1.77
dtype: float64

**Observation:** Sales amount is positively skewed. This means a smaller number of higher-value records pull the distribution upward, so median and percentiles are useful alongside the mean.

## 3. Correlation and Covariance

### Question: What are the linear relationships among Age, Quantity, and Amount?

In [5]:
df[["Age","Qty","Amount"]].corr()

,Age,Qty,Amount
Age,1.00,-0.00,0.00
Qty,-0.00,1.00,0.05
Amount,0.00,0.05,1.00


In [6]:
df[["Age","Qty","Amount"]].cov()

,Age,Qty,Amount
Age,228.34,-0.00,7.17
Qty,-0.00,0.09,4.26
Amount,7.17,4.26,"72,136.38"


**Observation:** Correlation measures standardized linear association, while covariance reflects the original units. These measures describe association only and do not establish causation.

## 4. Hypothesis Test – Channel and Order Status

### Business Question
Is order status distribution independent of sales channel?

**Null Hypothesis (H₀):** Order status and sales channel are independent.

**Alternative Hypothesis (H₁):** Order status and sales channel are associated.

**Significance level:** α = 0.05

In [7]:
ct = pd.crosstab(df["Channel "], df["Status"])
chi2, p, dof, expected = stats.chi2_contingency(ct)
pd.Series({"Chi-square statistic": chi2, "p-value": p, "Degrees of freedom": dof})

Chi-square statistic   10.13
p-value                 0.93
Degrees of freedom     18.00
dtype: float64

**Decision:** Reject H₀ if p-value < 0.05.

**Business Interpretation:** The test evaluates whether status mix differs across channels. A statistically significant result means channel and status are associated in this dataset; it does not prove that the channel causes the status outcome.

## 5. Hypothesis Test – Sales Amount by Gender

### Business Question
Do average sales amounts differ between Women and Men?

**H₀:** The mean sales amount is the same for Women and Men.

**H₁:** The mean sales amount differs between Women and Men.

**Test:** Welch's independent two-sample t-test.

**Significance level:** α = 0.05

In [8]:
women = df.loc[df["Gender"]=="Women", "Amount"]
men = df.loc[df["Gender"]=="Men", "Amount"]
t_stat, p_value = stats.ttest_ind(women, men, equal_var=False, nan_policy="omit")
pd.Series({"t-statistic": t_stat, "p-value": p_value,
           "Women mean": women.mean(), "Men mean": men.mean()})

t-statistic   -56.74
p-value         0.00
Women mean    628.31
Men mean      800.75
dtype: float64

**Decision:** Reject H₀ when p-value < 0.05.

**Business Interpretation:** The result tests whether the average line-item sales amount differs by gender. Any statistically significant difference should still be evaluated for practical size before making a business decision.

## 6. Hypothesis Test – Category Sales Amount

### Business Question
Do mean sales amounts differ across product categories?

**H₀:** All category means are equal.

**H₁:** At least one category mean differs.

**Test:** One-way ANOVA.

**Significance level:** α = 0.05

In [9]:
category_groups = [g["Amount"].values for _, g in df.groupby("Category")]
f_stat, p_value = stats.f_oneway(*category_groups)
pd.Series({"F-statistic": f_stat, "p-value": p_value})

F-statistic   2,881.44
p-value           0.00
dtype: float64

**Business Interpretation:** A significant ANOVA result indicates that average line-item sales amount is not the same across all categories. ANOVA does not by itself identify which pairs differ; that would require an appropriate post-hoc analysis.

## 7. Confidence Interval

### Question: What is the 95% confidence interval for the mean line-item sales amount?

In [10]:
mean_amount = df["Amount"].mean()
ci_low, ci_high = stats.t.interval(
    confidence=0.95,
    df=len(df["Amount"])-1,
    loc=mean_amount,
    scale=stats.sem(df["Amount"])
)
pd.Series({"Mean Amount": mean_amount, "95% CI Lower": ci_low, "95% CI Upper": ci_high})

Mean Amount    682.07
95% CI Lower   679.09
95% CI Upper   685.06
dtype: float64

**Observation:** The confidence interval gives a statistically estimated range for the population mean under the sampling assumptions. It should not be interpreted as saying that 95% of individual transactions fall inside the interval.

## 8. Feature Engineering

### Purpose of engineered features
Only features with a clear business purpose are created:

- **Unit_Price:** Amount divided by quantity, useful for price-per-unit comparisons.
- **Order_Value:** Total sales amount across all rows belonging to an order, useful for order-level analysis.
- **Order_Item_Count:** Number of line items in an order, useful for basket complexity.
- **Order_Qty:** Total quantity in an order, useful for basket size.
- **Customer_Order_Count:** Number of distinct orders associated with a customer ID, useful for repeat-purchase behavior.
- **Month_Num:** Numeric month, useful for chronological sorting.

In [11]:
eng = df.copy()
eng["Unit_Price"] = eng["Amount"] / eng["Qty"]

order_stats = eng.groupby("Order ID").agg(
    Order_Value=("Amount","sum"),
    Order_Item_Count=("SKU","size"),
    Order_Qty=("Qty","sum")
)
eng = eng.join(order_stats, on="Order ID")

customer_order_count = eng.groupby("Cust ID")["Order ID"].nunique()
eng["Customer_Order_Count"] = eng["Cust ID"].map(customer_order_count)

eng["Month_Num"] = pd.to_datetime(eng["Month"], format="%b", errors="coerce").dt.month

eng[["Unit_Price","Order_Value","Order_Item_Count","Order_Qty","Customer_Order_Count","Month_Num"]].head()

,Unit_Price,Order_Value,Order_Item_Count,Order_Qty,Customer_Order_Count,Month_Num
0,376.00,376.00,1.00,1.00,1.00,12.00
1,"1,449.00","1,449.00",1.00,1.00,1.00,12.00
2,453.00,453.00,1.00,1.00,1.00,12.00
3,729.00,729.00,1.00,1.00,1.00,12.00
4,544.00,544.00,1.00,1.00,1.00,12.00


## 9. Feature Evaluation

### Question: Do the engineered features add analytical value?

In [12]:
feature_evaluation = pd.DataFrame({
    "Feature": ["Unit_Price","Order_Value","Order_Item_Count","Order_Qty","Customer_Order_Count","Month_Num"],
    "Business_use": [
        "Compare price per unit when quantity differs",
        "Measure total value of an order",
        "Understand number of line items per order",
        "Measure total units in an order",
        "Identify repeat-purchase intensity",
        "Support chronological monthly analysis"
    ]
})
feature_evaluation

,Feature,Business_use
0,Unit_Price,Compare price per unit when quantity differs
1,Order_Value,Measure total value of an order
2,Order_Item_Count,Understand number of line items per order
3,Order_Qty,Measure total units in an order
4,Customer_Order_Count,Identify repeat-purchase intensity
5,Month_Num,Support chronological monthly analysis


**Observation:** The engineered features add order-, customer-, price-, and time-level views that are not directly available from the raw line-item fields. They support more meaningful business questions without changing the original measures.

## 10. Final Analytical Findings

In [13]:
final_findings = pd.Series({
    "Total sales": df["Amount"].sum(),
    "Orders": df["Order ID"].nunique(),
    "Customers": df["Cust ID"].nunique(),
    "Top category": df.groupby("Category")["Amount"].sum().idxmax(),
    "Top channel": df.groupby("Channel ")["Amount"].sum().idxmax(),
    "Top state": df.groupby("ship-state")["Amount"].sum().idxmax(),
    "Top month": df.groupby("Month")["Amount"].sum().idxmax()
})
final_findings

Total sales        21176377
Orders                27555
Customers             27490
Top category            Set
Top channel          Amazon
Top state       MAHARASHTRA
Top month               Mar
dtype: object

**Observation:** The final findings combine descriptive and engineered views. Recommendations should focus on high-performing categories/channels/states, operational status patterns, customer segments, and monthly sales variation, while avoiding unsupported causal claims.

## 11. Save Feature-Engineered Dataset

In [14]:
eng.to_csv("SwiftCart_Feature_Engineered_Dataset.csv", index=False)
print("Saved: SwiftCart_Feature_Engineered_Dataset.csv")

Saved: SwiftCart_Feature_Engineered_Dataset.csv
